# Environment check

Quick smoke test before doing anything serious. We want to know:
what GPU did Colab give us, does it support bf16, and are all the
libraries we need actually installable in this runtime.

Run this every time you start a new Colab session.

In [1]:
# What GPU did we get? Print the device, VRAM, CUDA version,
# and whether bf16 is supported. If this prints a T4 instead of
# an A100, change the runtime type from the menu and come back.
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('BF16 supported:', torch.cuda.is_bf16_supported())
print('CUDA version:', torch.version.cuda)
print('PyTorch version:', torch.__version__)

Mon May 25 18:35:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Install the libraries we need. Pinning to versions that worked
# for the May 2026 run. Newer versions probably work too but
# the TRL API changes often, so be careful.
# Unsloth is optional and only speeds things up; skip it if the
# install takes forever.
!pip install -q \
    'transformers>=4.53.0' \
    'trl>=0.18.0' \
    'peft>=0.14.0' \
    'bitsandbytes>=0.45.0' \
    'accelerate>=1.4.0' \
    'datasets>=3.0.0' \
    'wandb>=0.19.0' \
    'lm-eval>=0.4.5' \
    'vllm>=0.8.5' \
    'flash-attn --no-build-isolation'
# Unsloth (optional, speeds up training ~2x)
!pip install -q 'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'

ERROR: Invalid requirement: 'flash-attn --no-build-isolation': Expected end or semicolon (after name and no valid version specifier)
    flash-attn --no-build-isolation
               ^
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 146.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.6/869.6 kB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Log in to Hugging Face and Weights & Biases. The notebook will
# prompt for tokens. Make a write-scoped token at
# huggingface.co/settings/tokens and a wandb key at wandb.ai/authorize.
from huggingface_hub import login
login()  # paste your HF token with write permissions

import wandb
wandb.login()  # paste your W&B API key

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


False

In [4]:
# Sanity-check that the imports work and print their versions.
# If any of these fail with an ImportError, restart the runtime
# and re-run the install cell above.
import transformers, trl, peft, bitsandbytes, accelerate, datasets
for lib in [transformers, trl, peft, bitsandbytes, accelerate, datasets]:
    print(f'{lib.__name__}: {lib.__version__}')

transformers: 5.5.0
trl: 0.24.0
peft: 0.19.1
bitsandbytes: 0.49.2
accelerate: 1.13.0
datasets: 4.3.0


In [5]:
# Pick the right config file based on how much VRAM the GPU has.
# A100 80GB and H100 80GB use the bigger config; A100 40GB uses
# the smaller one. Anything below 38GB needs the fallback model.
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 75:
    print('A100 80GB detected — use configs/a100_80gb.yaml')
elif vram_gb >= 38:
    print('A100 40GB detected — use configs/a100_40gb.yaml')
else:
    print(f'Only {vram_gb:.1f}GB VRAM — use fallback model Qwen3-4B or Qwen2.5-Math-1.5B')

A100 40GB detected — use configs/a100_40gb.yaml


In [6]:
!pip install flash-attn --no-build-isolation -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 75.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Operation cancelled by user


In [7]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/drive/MyDrive/llm_posttraining/data', exist_ok=True)

# Copy everything saved in notebook 01 to Drive
shutil.copytree('/content/data/sft_train',    '/content/drive/MyDrive/llm_posttraining/data/sft_train')
shutil.copytree('/content/data/sft_val',      '/content/drive/MyDrive/llm_posttraining/data/sft_val')
shutil.copytree('/content/data/sft_test',     '/content/drive/MyDrive/llm_posttraining/data/sft_test')
shutil.copytree('/content/data/grpo_prompts', '/content/drive/MyDrive/llm_posttraining/data/grpo_prompts')
print('Data backed up to Drive.')

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/sft_train'